<a href="https://colab.research.google.com/github/eaedk/Machine-Learning-Tutorials-Updated/blob/main/03_computer_vision/03_object_detection_quest_yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Quest · Object detection: what is in the picture, and where?

Image classification answers *"what is this picture?"*. **Object detection** answers *"which objects are in this
picture, and where exactly?"*, by drawing a **bounding box** with a **class** and a **confidence** around each object.
It powers self-driving cars, wildlife camera traps, crop counting, retail shelves, medical imaging…

**A short history**
- **2001 · Viola-Jones.** The first real-time face detector (hand-crafted features + a cascade of classifiers). It
  ended up in every digital camera of the 2000s.
- **2014 · R-CNN.** Deep learning arrives: propose ~2 000 regions, classify each with a CNN. Accurate but very slow.
- **2015/2016 · YOLO, "You Only Look Once"** (Joseph Redmon et al.). One single network looks at the image **once** and
  predicts all boxes at the same time, in real time. A revolution.
- **2020 → today.** YOLO becomes a family: YOLOv5, YOLOv8 and YOLO11 by Ultralytics, and the newest generation,
  **YOLO26**, designed to be fast on small devices (phones, Raspberry Pi, drones). The latest versions are trained to
  output one box per object, so the classic post-processing step (NMS, see below) is no longer required at inference.

**Your mission:** master the building blocks (box formats, IoU, NMS, mAP), then use and fine-tune a YOLO26 model.

| You will practise | Concept |
|---|---|
| Reading YOLO labels | box formats, normalised coordinates |
| Measuring overlap | **IoU** (Intersection over Union), coded by you |
| Removing duplicates | **NMS** (Non-Maximum Suppression), coded by you |
| Scoring a detector | precision, recall, **mAP** |
| Using & fine-tuning YOLO26 | Ultralytics API, dataset YAML |

In [ ]:
# ⚙️ Setup: run me first! Finds the course toolkit `mlquest` (Windows, macOS, Linux and Colab).
import os, sys, subprocess, pathlib
if "google.colab" in sys.modules:  # on Colab: grab the repo (once) and the few missing packages
    if not pathlib.Path("/content/Machine-Learning-Tutorials-Updated").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/eaedk/Machine-Learning-Tutorials-Updated", "/content/Machine-Learning-Tutorials-Updated"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    os.chdir("/content/Machine-Learning-Tutorials-Updated")
ROOT = pathlib.Path.cwd()
while not (ROOT / "mlquest").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mlquest import *  # paths, seed_everything, get_device, quest/checkpoint, Leaderboard...
set_style()
SEED = seed_everything(42)

In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

os.environ["YOLO_VERBOSE"] = "False"          # quiet logs. Set to "True" (before this import) to watch every step
from ultralytics import YOLO
from ultralytics.utils import ASSETS, YAML
from ultralytics.utils.checks import check_yaml

DEVICE = get_device()
YOLO_DATA = DATA_DIR / "yolo_datasets"
WEIGHTS = DATA_DIR / "weights"
WEIGHTS.mkdir(exist_ok=True)


def yolo_dataset(yaml_name: str) -> Path:
    """Download an Ultralytics dataset into data/yolo_datasets/ and return a YAML pointing to it."""
    cfg = YAML.load(check_yaml(f"{yaml_name}.yaml"))
    folder = YOLO_DATA / Path(cfg["path"]).name
    if not folder.exists():
        url = cfg["download"]
        stem = Path(url).stem
        archive = download(url, YOLO_DATA / Path(url).name)
        tmp = extract(archive, YOLO_DATA / f"_tmp_{stem}")
        inner = tmp / stem if (tmp / stem).is_dir() else tmp
        shutil.move(str(inner), str(folder))
        shutil.rmtree(tmp, ignore_errors=True)
        archive.unlink()
    cfg["path"] = str(folder)
    cfg.pop("download", None)
    out = YOLO_DATA / f"{yaml_name}.yaml"
    YAML.save(out, cfg)
    return out

quest("What is in the picture, and where?", chapter="Track 3 · Computer vision · Object detection", xp=300)

## 1. Boxes and the YOLO label format 📦

We use **COCO8**: the first 8 images of COCO, the reference detection dataset (330 000 images, 80 classes, created by
Microsoft in 2014). It's tiny on purpose: perfect to understand the format.

A YOLO dataset is two folders: `images/` and `labels/`. For each image `xxx.jpg` there is a text file `xxx.txt` with
**one line per object**:

```
class_id  x_center  y_center  width  height        ← all normalised between 0 and 1 (divided by image width/height)
```

Normalised coordinates don't depend on the image size: resize the image, the labels stay valid.

In [ ]:
coco8_yaml = yolo_dataset("coco8")
coco8 = YOLO_DATA / "coco8"
NAMES = YAML.load(coco8_yaml)["names"]
img_path = sorted((coco8 / "images" / "train").glob("*.jpg"))[1]
label_path = coco8 / "labels" / "train" / (img_path.stem + ".txt")
print(label_path.read_text())
img = Image.open(img_path)
W, H = img.size
print("image size:", W, "x", H)

### 🎯 Exercise 1: convert a YOLO label to pixel corners
Write `yolo_to_xyxy(line, W, H)` that returns `(class_id, x1, y1, x2, y2)` in **pixels** (top-left and bottom-right
corners).

In [ ]:
def yolo_to_xyxy(line: str, W: int, H: int):
    cls, xc, yc, w, h = line.split()
    cls, xc, yc, w, h = int(cls), float(xc) * W, float(yc) * H, float(w) * W, float(h) * H
    x1 = ...   # TODO: from the centre, go left by half the width
    y1 = ...   # TODO
    x2 = ...   # TODO
    y2 = ...   # TODO
    return cls, x1, y1, x2, y2

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def yolo_to_xyxy(line: str, W: int, H: int):
    cls, xc, yc, w, h = line.split()
    cls, xc, yc, w, h = int(cls), float(xc) * W, float(yc) * H, float(w) * W, float(h) * H
    x1 = xc - w / 2
    y1 = yc - h / 2
    x2 = xc + w / 2
    y2 = yc + h / 2
    return cls, x1, y1, x2, y2
```

</details>

In [ ]:
check(np.allclose(yolo_to_xyxy("0 0.5 0.5 0.2 0.4", 100, 200)[1:], (40, 60, 60, 140)),
      "Correct! Let's draw the boxes of this image with your function.",
      "For '0 0.5 0.5 0.2 0.4' on a 100×200 image you should get (0, 40, 60, 60, 140).")

def draw_boxes(img, boxes, title="", color="lime"):
    """boxes: list of (label_text, x1, y1, x2, y2)."""
    fig, ax = plt.subplots(figsize=(6, 6 * img.size[1] / img.size[0]))
    ax.imshow(img); ax.axis("off"); ax.set_title(title)
    for text, x1, y1, x2, y2 in boxes:
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, lw=2))
        ax.text(x1, y1 - 4, text, color="black", backgroundcolor=color, fontsize=8)
    plt.show()

gt = [yolo_to_xyxy(l, W, H) for l in label_path.read_text().splitlines()]
draw_boxes(img, [(NAMES[c], *b) for c, *b in gt], "Ground-truth boxes (from the label file)")

## 2. IoU: how much do two boxes overlap? 📐

To decide if a predicted box is correct, we compare it with the true box using **Intersection over Union**:

```
          area of the overlap
IoU = ─────────────────────────────          0 = no overlap · 1 = perfect match
       area of the union of the two
```

A prediction usually counts as correct if **IoU ≥ 0.5** with a true box of the same class.

### 🎯 Exercise 2: implement IoU for two boxes in `(x1, y1, x2, y2)` format

In [ ]:
def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])     # top-left corner of the intersection
    ix2, iy2 = ...                                 # TODO: bottom-right corner of the intersection
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)  # 0 if the boxes don't overlap
    area_a = ...                                   # TODO
    area_b = ...                                   # TODO
    return inter / (area_a + area_b - inter)

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter)
```

</details>

In [ ]:
from torchvision.ops import box_iou
tests = [((0, 0, 10, 10), (0, 0, 10, 10)), ((0, 0, 10, 10), (5, 0, 15, 10)), ((0, 0, 10, 10), (20, 20, 30, 30))]
mine = [iou(a, b) for a, b in tests]
ref = [box_iou(torch.tensor([a], dtype=torch.float), torch.tensor([b], dtype=torch.float)).item() for a, b in tests]
print("yours:", np.round(mine, 3), " torchvision:", np.round(ref, 3))
check(np.allclose(mine, ref), "Your IoU matches torchvision's. Identical → 1, half-shifted → 1/3, apart → 0.",
      "Compare your values with torchvision's. Did you use min() for the bottom-right corner?")
checkpoint("detection/iou", xp=50)

## 3. NMS: one object, one box 🧹

Classic detectors predict **many overlapping boxes** for the same object. **Non-Maximum Suppression** cleans up:
1. sort boxes by confidence,
2. keep the most confident one, delete every remaining box that overlaps it with IoU > threshold,
3. repeat with the next remaining box.

### 🎯 Exercise 3: implement greedy NMS

In [ ]:
rng = np.random.default_rng(SEED)
objects = [(60, 50, 200, 220), (230, 80, 380, 260)]           # 2 real objects
cand, scores = [], []
for x1, y1, x2, y2 in objects:                               # a detector fires 6 noisy boxes per object
    for _ in range(6):
        j = rng.normal(0, 8, 4)
        cand.append((x1 + j[0], y1 + j[1], x2 + j[2], y2 + j[3])); scores.append(rng.uniform(0.5, 0.95))
canvas = Image.new("RGB", (420, 300), "white")
draw_boxes(canvas, [(f"{s:.2f}", *b) for b, s in zip(cand, scores)], "12 raw predictions for 2 objects", color="orange")

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    order = list(np.argsort(scores)[::-1])     # indices, most confident first
    keep = []
    while order:
        best = order.pop(0)
        keep.append(best)
        order = ...   # TODO: keep only the indices i in order whose iou(boxes[best], boxes[i]) <= iou_threshold
    return keep

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def nms(boxes, scores, iou_threshold=0.5):
    order = list(np.argsort(scores)[::-1])
    keep = []
    while order:
        best = order.pop(0)
        keep.append(best)
        order = [i for i in order if iou(boxes[best], boxes[i]) <= iou_threshold]
    return keep
```

</details>

In [ ]:
from torchvision.ops import nms as tv_nms
kept = nms(cand, scores)
ref = tv_nms(torch.tensor(cand, dtype=torch.float), torch.tensor(scores, dtype=torch.float), 0.5).tolist()
check(sorted(kept) == sorted(ref) and len(kept) == 2,
      "2 boxes left for 2 objects, same result as torchvision. NMS cleaned the mess!",
      f"Expected the same indices as torchvision: {sorted(ref)}")
draw_boxes(canvas, [(f"{scores[i]:.2f}", *cand[i]) for i in kept], "After NMS")

## 4. How do we score a detector? mAP 🏅

For each class, sort all predictions by confidence and walk down the list:
- **Precision** = share of predictions that are correct (IoU ≥ 0.5 with an unmatched true box),
- **Recall** = share of true objects that were found.

Lowering the confidence threshold finds more objects (recall ↑) but adds false alarms (precision ↓). The area under
the precision-recall curve is the **Average Precision (AP)**. The mean over classes is the **mAP**.

- **mAP50**: AP with the IoU threshold 0.5 ("is the box roughly right?").
- **mAP50-95**: average of the mAP over IoU thresholds 0.50, 0.55 … 0.95 (rewards *precise* boxes). It is the main
  COCO metric, and the one most detection competitions use.

## 5. YOLO26 out of the box 🚀

YOLO26n ("n" = nano, 2.4 M parameters) was pre-trained on COCO's 80 classes. Let's see what it finds in a
famous Ultralytics test image. Weights are downloaded once into `data/weights/`.

In [ ]:
model = YOLO(str(WEIGHTS / "yolo26n.pt"))
results = model.predict(ASSETS / "bus.jpg", conf=0.25, device=DEVICE)
r = results[0]
plt.figure(figsize=(6, 8)); plt.imshow(r.plot()[..., ::-1]); plt.axis("off"); plt.title("YOLO26n predictions"); plt.show()
pd.DataFrame({"class": [r.names[int(c)] for c in r.boxes.cls], "confidence": r.boxes.conf.cpu().numpy().round(2)})

### 🎯 Exercise 4: count the people
`r.boxes.cls` holds the class id of each box (`0` = person in COCO), `r.boxes.conf` the confidence.
Count the persons detected with a confidence above 0.5.

In [ ]:
n_people = ...   # TODO: hint: ((r.boxes.cls == 0) & (r.boxes.conf > 0.5)).sum().item()
print("People in the picture:", n_people)

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
n_people = ((r.boxes.cls == 0) & (r.boxes.conf > 0.5)).sum().item()
print("People in the picture:", n_people)
```

</details>

In [ ]:
check(isinstance(n_people, int) and n_people >= 3, f"{n_people} people found. You just built a people counter!",
      "Filter the boxes on class 0 and confidence > 0.5, then sum.")
checkpoint("detection/inference", xp=50)

## 6. Fine-tuning on your own data 🏋️

To detect *your* objects (animals, weeds, potholes…), you fine-tune a pre-trained model on your labelled images.
Ultralytics only needs a small **dataset YAML** describing where images are and the class names:

In [ ]:
print(Path(coco8_yaml).read_text()[:400])

### 🎯 Exercise 5: fine-tune YOLO26n on COCO8 for 10 epochs
Use `model.train(...)` with `data=str(coco8_yaml)`, `epochs=10`, `imgsz=320`, `batch=8`, `device=DEVICE`,
`workers=NUM_WORKERS`, and save the run in our runs folder with `project=str(RUNS_DIR / "detection_quest")`,
`name="coco8"`, `exist_ok=True`.

(8 images is far too few to really learn: here we only rehearse the workflow. The next notebook trains on 1 000
wildlife photos.)

In [ ]:
ft_model = YOLO(str(WEIGHTS / "yolo26n.pt"))
metrics = ...   # TODO: ft_model.train(...)
print(f"mAP50 = {metrics.box.map50:.3f} · mAP50-95 = {metrics.box.map:.3f}")

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
ft_model = YOLO(str(WEIGHTS / "yolo26n.pt"))
metrics = ft_model.train(data=str(coco8_yaml), epochs=10, imgsz=320, batch=8, device=DEVICE, workers=NUM_WORKERS,
                         project=str(RUNS_DIR / "detection_quest"), name="coco8", exist_ok=True,
                         plots=False, deterministic=False)
print(f"mAP50 = {metrics.box.map50:.3f} · mAP50-95 = {metrics.box.map:.3f}")
```

</details>

In [ ]:
curves = pd.read_csv(Path(ft_model.trainer.save_dir) / "results.csv")
curves.columns = curves.columns.str.strip()
curves.plot(x="epoch", y=["metrics/mAP50(B)", "metrics/mAP50-95(B)"], marker="o", title="Validation mAP per epoch")
plt.show()
check(metrics.box.map50 > 0.3, "Training works end to end. You can now fine-tune YOLO on any dataset!",
      "mAP looks too low: check the arguments of train().")

## 🏁 Quest complete!

You can read labels, compute IoU, run NMS, interpret mAP and fine-tune a state-of-the-art detector.

### 🚀 Your turn (bonus XP)
1. Lower `conf` to 0.05 in the bus prediction: what appears? Raise it to 0.8?
2. Use `model.predict(..., classes=[0])` to detect only people.
3. Plot a precision-recall curve yourself from the predictions of the fine-tuned model on COCO8.
4. Try `model.track(source=<a video file>)` on a video of your street: detection + tracking IDs!

Next: `04_object_detection_app_african_wildlife.ipynb`, a real fine-tuning on African wildlife.

In [ ]:
checkpoint("detection/complete", xp=200)
badge("Box Master", "🎯")
progress()